# Common Crawl Collection Reporting

This notebook collates the small set of Common Crawl collection numbers that are useful for the Materials section of the dissertation. It reads the configured crawl years, final processed trend output, accepted corpus quality summaries, processed corpus document file, and throughput summaries.

The only saved output is `reports/tables/commoncrawl_collection_reporting.csv`. The notebook still performs consistency checks before writing that file, so stale or incomplete synced artifacts fail loudly rather than producing misleading paper numbers.

In [16]:
from pathlib import Path

import pandas as pd
import yaml

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "configs/commoncrawl_collection.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "configs/commoncrawl_collection.yaml").exists():
    raise FileNotFoundError("Run this notebook from the repository root or from notebooks/.")

CONFIG_PATH = REPO_ROOT / "configs/commoncrawl_collection.yaml"
INTERIM_COLLECTION_DIR = REPO_ROOT / "data/interim/collection"
PROCESSED_DIR = REPO_ROOT / "data/processed"
REPORTING_CSV_PATH = REPO_ROOT / "reports/tables/summary/commoncrawl_collection_reporting.csv"

## 1. Reporting Frame

The configured crawl map defines the expected reporting frame. The instance type is recorded from the public config so throughput can be described in relation to the machine used for collection.

In [17]:
with CONFIG_PATH.open() as fh:
    config = yaml.safe_load(fh)

crawl_map = pd.DataFrame(config["collection"]["crawl_map"])
configured_years = sorted(crawl_map["year"].astype(int).tolist())
year_min = min(configured_years)
year_max = max(configured_years)
year_count = len(configured_years)
instance_type = config["collection"].get("aws", {}).get("ec2", {}).get("instance_type", "unknown")

crawl_map

,year,crawl_id
0,2014,CC-MAIN-2014-15
1,2015,CC-MAIN-2015-18
2,2016,CC-MAIN-2016-18
3,2017,CC-MAIN-2017-17
4,2018,CC-MAIN-2018-17
5,2019,CC-MAIN-2019-18
6,2020,CC-MAIN-2020-16
7,2021,CC-MAIN-2021-17
8,2022,CC-MAIN-2022-21
9,2023,CC-MAIN-2023-14


## 2. Small Helpers

These helpers keep the notebook compact while preserving the main reporting logic in plain, inspectable cells below.

In [18]:
def read_metric_series(path):
    return pd.read_csv(path).set_index("metric")["value"]


def metric_number(metrics, name, default=0):
    if name not in metrics.index:
        return default
    value = pd.to_numeric(pd.Series([metrics.loc[name]]), errors="coerce").iloc[0]
    if pd.isna(value):
        return default
    return int(value) if float(value).is_integer() else float(value)


def latest_batch_rows(base_dir, filename_glob):
    rows = []
    for path in sorted(base_dir.glob(f"*/batch_*/*/{filename_glob}")):
        year_part, batch_part, runid_part, _ = path.relative_to(base_dir).parts
        rows.append(
            {
                "year": int(year_part),
                "batch": batch_part,
                "batch_number": int(batch_part.replace("batch_", "")),
                "runid": runid_part,
                "path": path,
            }
        )
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise FileNotFoundError(f"No files matching {filename_glob} found under {base_dir}")
    return (
        frame.sort_values(["year", "batch_number", "runid"])
        .drop_duplicates(["year", "batch_number"], keep="last")
        .reset_index(drop=True)
    )


def hours_per_million(elapsed_hours, wet_records_scanned):
    if wet_records_scanned == 0:
        return pd.NA
    return elapsed_hours / (wet_records_scanned / 1_000_000)

## 3. Trend Track

Trend counts come from the final processed trend file, because the processed builder enforces exactly one row per configured year. Throughput comes from the accepted per-year throughput summaries.

In [19]:
trend_rates_path = PROCESSED_DIR / "trend/trend_rates.csv"
trend_rates = pd.read_csv(trend_rates_path)
trend_rates["year"] = trend_rates["year"].astype(int)
trend_reporting_rates = trend_rates.copy()
if "aggregation_level" in trend_reporting_rates.columns:
    trend_reporting_rates = trend_reporting_rates.loc[trend_reporting_rates["aggregation_level"].eq("all")].copy()

trend_metric_dir = INTERIM_COLLECTION_DIR / "metrics/trend"
trend_throughput_files = latest_batch_rows(trend_metric_dir, "cc_collection_throughput_summary_*.csv")

trend_throughput_records = []
for row in trend_throughput_files.itertuples(index=False):
    metrics = read_metric_series(row.path)
    trend_throughput_records.append(
        {
            "year": row.year,
            "batch_number": row.batch_number,
            "total_elapsed_hours": metric_number(metrics, "total_observed_elapsed_sec") / 3600,
            "wet_elapsed_hours": metric_number(metrics, "wet.elapsed_sec") / 3600,
            "warc_elapsed_hours": metric_number(metrics, "warc.elapsed_sec") / 3600,
            "document_quality_elapsed_hours": metric_number(metrics, "document_quality.elapsed_sec") / 3600,
            "warc_fetch_success_rate_pct": metric_number(metrics, "warc.fetch_success_rate_pct"),
            "warc_extract_success_rate_pct": metric_number(metrics, "warc.extract_success_rate_pct"),
        }
    )

trend_throughput = pd.DataFrame(trend_throughput_records)
trend_by_year = trend_reporting_rates.merge(trend_throughput, on="year", how="left")
trend_by_year = trend_by_year.rename(
    columns={
        "docs_scanned": "wet_records_scanned",
        "validated_hits_wet": "wet_validated_hits",
        "validated_hits_warc": "warc_validated_hits",
    }
)
trend_by_year["track"] = "trend"
trend_by_year["row_type"] = "year"
trend_by_year["batch_count"] = 1
trend_by_year["final_documents"] = pd.NA
trend_by_year["target_documents"] = pd.NA
trend_by_year["adhd_documents"] = pd.NA
trend_by_year["autism_documents"] = pd.NA
trend_by_year["hours_per_million_wet_records"] = trend_by_year.apply(
    lambda row: hours_per_million(row["total_elapsed_hours"], row["wet_records_scanned"]), axis=1
)

trend_by_year[
    [
        "track",
        "year",
        "wet_records_scanned",
        "warc_validated_hits",
        "total_elapsed_hours",
        "hours_per_million_wet_records",
    ]
]

,track,year,wet_records_scanned,warc_validated_hits,total_elapsed_hours,hours_per_million_wet_records
0,trend,2014,2654179.0,7241.0,6.153888,2.318566
1,trend,2015,2595430.0,6128.0,2.133771,0.822126
2,trend,2016,2842421.0,10439.0,3.388940,1.192272
3,trend,2017,2215705.0,5986.0,1.878133,0.847646
4,trend,2018,2346807.0,6164.0,1.767904,0.753323
5,trend,2019,2187679.0,6924.0,2.457539,1.123355
6,trend,2020,2509938.0,6913.0,2.530095,1.008031
7,trend,2021,2415283.0,7120.0,2.806042,1.161786
8,trend,2022,2111324.0,5984.0,1.844984,0.873851
9,trend,2023,1899490.0,4932.0,1.460020,0.768638


## 4. Corpus Track

Corpus counts and throughput are read from the latest accepted summary for each year/batch. This remains valid after expansion batches are added.

In [20]:
corpus_quality_dir = INTERIM_COLLECTION_DIR / "quality/corpus"
corpus_metric_dir = INTERIM_COLLECTION_DIR / "metrics/corpus"
corpus_summary_files = latest_batch_rows(corpus_quality_dir, "cc_collection_summary_*.csv")
corpus_throughput_files = latest_batch_rows(corpus_metric_dir, "cc_collection_throughput_summary_*.csv")

corpus_records = []
for row in corpus_summary_files.itertuples(index=False):
    metrics = read_metric_series(row.path)
    corpus_records.append(
        {
            "year": row.year,
            "batch_number": row.batch_number,
            "runid": row.runid,
            "wet_records_scanned": metric_number(metrics, "docs_scanned"),
            "wet_validated_hits": metric_number(metrics, "validated_hits_wet"),
            "warc_validated_hits": metric_number(metrics, "validated_hits_warc"),
            "final_documents": metric_number(metrics, "final.doc_count"),
            "target_documents": metric_number(metrics, "final.term_role.target.doc_count"),
            "adhd_documents": metric_number(metrics, "final.term_group.adhd.doc_count"),
            "autism_documents": metric_number(metrics, "final.term_group.autism.doc_count"),
        }
    )

corpus_throughput_records = []
for row in corpus_throughput_files.itertuples(index=False):
    metrics = read_metric_series(row.path)
    corpus_throughput_records.append(
        {
            "year": row.year,
            "batch_number": row.batch_number,
            "total_elapsed_hours": metric_number(metrics, "total_observed_elapsed_sec") / 3600,
            "wet_elapsed_hours": metric_number(metrics, "wet.elapsed_sec") / 3600,
            "warc_elapsed_hours": metric_number(metrics, "warc.elapsed_sec") / 3600,
            "document_quality_elapsed_hours": metric_number(metrics, "document_quality.elapsed_sec") / 3600,
            "warc_fetch_success_rate_pct": metric_number(metrics, "warc.fetch_success_rate_pct"),
            "warc_extract_success_rate_pct": metric_number(metrics, "warc.extract_success_rate_pct"),
        }
    )

corpus_batches = pd.DataFrame(corpus_records).merge(
    pd.DataFrame(corpus_throughput_records), on=["year", "batch_number"], how="left"
)
corpus_missing_throughput = corpus_batches.loc[
    corpus_batches["total_elapsed_hours"].isna(), ["year", "batch_number"]
].copy()

corpus_by_year = (
    corpus_batches.groupby("year", as_index=False)
    .agg(
        batch_count=("batch_number", "nunique"),
        wet_records_scanned=("wet_records_scanned", "sum"),
        wet_validated_hits=("wet_validated_hits", "sum"),
        warc_validated_hits=("warc_validated_hits", "sum"),
        final_documents=("final_documents", "sum"),
        target_documents=("target_documents", "sum"),
        adhd_documents=("adhd_documents", "sum"),
        autism_documents=("autism_documents", "sum"),
        throughput_batch_count=("total_elapsed_hours", "count"),
        total_elapsed_hours=("total_elapsed_hours", "sum"),
        wet_elapsed_hours=("wet_elapsed_hours", "sum"),
        warc_elapsed_hours=("warc_elapsed_hours", "sum"),
        document_quality_elapsed_hours=("document_quality_elapsed_hours", "sum"),
        warc_fetch_success_rate_pct=("warc_fetch_success_rate_pct", "min"),
        warc_extract_success_rate_pct=("warc_extract_success_rate_pct", "min"),
    )
)
corpus_by_year["track"] = "corpus"
corpus_by_year["row_type"] = "year"
corpus_by_year["throughput_complete"] = corpus_by_year["throughput_batch_count"].eq(
    corpus_by_year["batch_count"]
)
corpus_by_year["hours_per_million_wet_records"] = corpus_by_year.apply(
    lambda row: hours_per_million(row["total_elapsed_hours"], row["wet_records_scanned"])
    if row["throughput_complete"]
    else pd.NA,
    axis=1,
)
corpus_incomplete_throughput = ~corpus_by_year["throughput_complete"]
corpus_by_year.loc[
    corpus_incomplete_throughput,
    [
        "total_elapsed_hours",
        "wet_elapsed_hours",
        "warc_elapsed_hours",
        "document_quality_elapsed_hours",
        "warc_fetch_success_rate_pct",
        "warc_extract_success_rate_pct",
    ],
] = pd.NA

corpus_by_year[
    [
        "track",
        "year",
        "batch_count",
        "wet_records_scanned",
        "final_documents",
        "target_documents",
        "total_elapsed_hours",
        "hours_per_million_wet_records",
    ]
]

,track,year,batch_count,wet_records_scanned,final_documents,target_documents,total_elapsed_hours,hours_per_million_wet_records
0,corpus,2014,3,8222757,12124,3360,8.051696,0.979197
1,corpus,2015,4,10531523,13534,3613,8.557032,0.812516
2,corpus,2016,3,8768306,17471,4015,10.855511,1.23804
3,corpus,2017,4,8871512,12693,3262,7.882667,0.888537
4,corpus,2018,5,11816293,15738,4075,9.905514,0.838293
5,corpus,2019,3,6562726,10618,3273,NaN,<NA>
6,corpus,2020,3,7450904,10895,2917,8.228762,1.104398
7,corpus,2021,4,9660307,14629,3702,9.395262,0.972564
8,corpus,2022,4,8450550,13497,3368,7.958567,0.941781
9,corpus,2023,4,7616698,11235,2902,6.385049,0.838296


## 5. Processed Corpus Check

The final corpus parquet is checked against the accepted batch summaries. This catches stale processed outputs after reruns or expansion batches.

In [21]:
processed_corpus_path = PROCESSED_DIR / "corpus/corpus_documents.parquet"
processed_corpus = pd.read_parquet(
    processed_corpus_path,
    columns=["crawl_id", "url", "term_roles", "source_corpus_path"],
)

processed_duplicate_url_count = int(processed_corpus.duplicated(["crawl_id", "url"]).sum())
role_text = processed_corpus["term_roles"].fillna("").astype(str)
processed_target_document_count = int(role_text.str.contains(r"(?:^|\|)target(?:$|\|)", regex=True).sum())
processed_baseline_document_count = int(role_text.str.contains(r"(?:^|\|)baseline(?:$|\|)", regex=True).sum())
processed_source_count = int(processed_corpus["source_corpus_path"].nunique())

processed_summary = pd.DataFrame(
    [
        {
            "processed_rows": len(processed_corpus),
            "duplicate_crawl_url_rows": processed_duplicate_url_count,
            "duplicate_crawl_url_row_share": processed_duplicate_url_count / len(processed_corpus),
            "target_documents": processed_target_document_count,
            "baseline_documents": processed_baseline_document_count,
            "source_corpus_paths": processed_source_count,
        }
    ]
)

processed_summary

,processed_rows,duplicate_crawl_url_rows,duplicate_crawl_url_row_share,target_documents,baseline_documents,source_corpus_paths
0,167520,56,0.000334,43379,129046,54


## 6. Checks

These checks protect the paper numbers from partial syncs, stale processed outputs, and accidental duplicate years. Minor duplicate URL residue and missing timing summaries are reported as warnings rather than hard failures.

In [22]:
trend_years = sorted(trend_reporting_rates["year"].tolist())
corpus_years = sorted(corpus_by_year["year"].tolist())
trend_duplicate_years = sorted(trend_reporting_rates.loc[trend_reporting_rates["year"].duplicated(), "year"].unique().tolist())
processed_duplicate_url_share = processed_duplicate_url_count / len(processed_corpus)

checks = pd.DataFrame(
    [
        {
            "check": "trend years match configured crawl map",
            "passed": trend_years == configured_years,
            "detail": f"found={trend_years}; expected={configured_years}",
        },
        {
            "check": "trend has no duplicate years",
            "passed": not trend_duplicate_years,
            "detail": f"duplicate_years={trend_duplicate_years}",
        },
        {
            "check": "corpus summaries cover every configured year",
            "passed": corpus_years == configured_years,
            "detail": f"found={corpus_years}; expected={configured_years}",
        },
        {
            "check": "corpus batch summaries match processed corpus sources",
            "passed": len(corpus_batches) == processed_source_count,
            "detail": f"summary_batches={len(corpus_summary_files)}; processed_sources={processed_source_count}",
        },
        {
            "check": "processed corpus row count matches accepted quality summaries",
            "passed": len(processed_corpus) == int(corpus_by_year["final_documents"].sum()),
            "detail": f"processed={len(processed_corpus)}; summaries={int(corpus_by_year['final_documents'].sum())}",
        },
        {
            "check": "processed target document count matches accepted quality summaries",
            "passed": processed_target_document_count == int(corpus_by_year["target_documents"].sum()),
            "detail": f"processed={processed_target_document_count}; summaries={int(corpus_by_year['target_documents'].sum())}",
        },
    ]
)

if not checks["passed"].all():
    display(checks)
    failed = checks.loc[~checks["passed"], "check"].tolist()
    raise AssertionError(f"Collection reporting checks failed: {failed}")

warnings = pd.DataFrame(
    [
        {
            "warning": "trend throughput coverage",
            "active": sorted(trend_throughput["year"].tolist()) != configured_years,
            "detail": f"found={sorted(trend_throughput['year'].tolist())}; expected={configured_years}",
        },
        {
            "warning": "corpus throughput coverage",
            "active": not corpus_missing_throughput.empty,
            "detail": f"missing_batch_timings={corpus_missing_throughput.to_dict('records')}",
        },
        {
            "warning": "processed corpus duplicate crawl_id/url residue",
            "active": processed_duplicate_url_count > 0,
            "detail": f"duplicate_rows={processed_duplicate_url_count}; share={processed_duplicate_url_share:.6%}",
        },
    ]
)

if warnings["active"].any():
    display(warnings.loc[warnings["active"]])

checks

,warning,active,detail
0,trend throughput coverage,True,"found=[2014, 2015, 2016, 2017, 2018, 2019, 202..."
1,corpus throughput coverage,True,"missing_batch_timings=[{'year': 2019, 'batch_n..."
2,processed corpus duplicate crawl_id/url residue,True,duplicate_rows=56; share=0.033429%


,check,passed,detail
0,trend years match configured crawl map,True,"found=[2014, 2015, 2016, 2017, 2018, 2019, 202..."
1,trend has no duplicate years,True,duplicate_years=[]
2,corpus summaries cover every configured year,True,"found=[2014, 2015, 2016, 2017, 2018, 2019, 202..."
3,corpus batch summaries match processed corpus ...,True,summary_batches=54; processed_sources=54
4,processed corpus row count matches accepted qu...,True,processed=167520; summaries=167520
5,processed target document count matches accept...,True,processed=43379; summaries=43379


## 7. Single Reporting CSV

The CSV contains one topline row per track and one annual row per track/year. The normalized throughput measure is `hours_per_million_wet_records`, computed from total observed elapsed time divided by WET records scanned. This is the number to use for a concise runtime statement.

In [23]:
report_columns = [
    "row_type",
    "track",
    "year",
    "year_count",
    "year_range",
    "batch_count",
    "instance_type",
    "wet_records_scanned",
    "wet_validated_hits",
    "warc_validated_hits",
    "final_documents",
    "target_documents",
    "adhd_documents",
    "autism_documents",
    "total_elapsed_hours",
    "hours_per_million_wet_records",
    "throughput_observed_wet_records_scanned",
    "throughput_observed_year_count",
    "throughput_complete",
    "wet_elapsed_hours",
    "warc_elapsed_hours",
    "document_quality_elapsed_hours",
    "warc_fetch_success_rate_pct",
    "warc_extract_success_rate_pct",
    "report_note",
]

year_rows = pd.concat([trend_by_year, corpus_by_year], ignore_index=True)
year_rows["year_count"] = pd.NA
year_rows["year_range"] = pd.NA
year_rows["instance_type"] = instance_type
year_rows["throughput_complete"] = year_rows["throughput_complete"].where(
    year_rows["throughput_complete"].notna(), year_rows["total_elapsed_hours"].notna()
).astype(bool)
year_rows["throughput_observed_year_count"] = year_rows["throughput_complete"].astype(int)
year_rows["throughput_observed_wet_records_scanned"] = year_rows["wet_records_scanned"].where(
    year_rows["throughput_complete"], pd.NA
)
year_rows["report_note"] = "Annual accepted collection output."

topline_rows = []
for track, frame in year_rows.groupby("track", sort=False):
    total_wet = int(frame["wet_records_scanned"].sum())
    throughput_frame = frame.loc[frame["throughput_complete"]].copy()
    observed_wet = int(throughput_frame["wet_records_scanned"].sum())
    observed_year_count = int(throughput_frame["year"].nunique())
    throughput_complete = observed_year_count == year_count
    total_elapsed = float(throughput_frame["total_elapsed_hours"].sum())
    topline_rows.append(
        {
            "row_type": "track_total",
            "track": track,
            "year": pd.NA,
            "year_count": year_count,
            "year_range": f"{year_min}-{year_max}",
            "batch_count": int(frame["batch_count"].sum()),
            "instance_type": instance_type,
            "wet_records_scanned": total_wet,
            "wet_validated_hits": int(frame["wet_validated_hits"].sum()),
            "warc_validated_hits": int(frame["warc_validated_hits"].sum()),
            "final_documents": int(frame["final_documents"].sum()) if track == "corpus" else pd.NA,
            "target_documents": int(frame["target_documents"].sum()) if track == "corpus" else pd.NA,
            "adhd_documents": int(frame["adhd_documents"].sum()) if track == "corpus" else pd.NA,
            "autism_documents": int(frame["autism_documents"].sum()) if track == "corpus" else pd.NA,
            "total_elapsed_hours": total_elapsed,
            "hours_per_million_wet_records": hours_per_million(total_elapsed, observed_wet),
            "throughput_observed_wet_records_scanned": observed_wet,
            "throughput_observed_year_count": observed_year_count,
            "throughput_complete": throughput_complete,
            "wet_elapsed_hours": float(throughput_frame["wet_elapsed_hours"].sum()),
            "warc_elapsed_hours": float(throughput_frame["warc_elapsed_hours"].sum()),
            "document_quality_elapsed_hours": float(throughput_frame["document_quality_elapsed_hours"].sum()),
            "warc_fetch_success_rate_pct": float(throughput_frame["warc_fetch_success_rate_pct"].min()),
            "warc_extract_success_rate_pct": float(throughput_frame["warc_extract_success_rate_pct"].min()),
            "report_note": "Fixed-effort annual trend track." if track == "trend" else "Quality-gated corpus track.",
        }
    )

collection_reporting = pd.concat([pd.DataFrame(topline_rows), year_rows], ignore_index=True)
collection_reporting = collection_reporting[report_columns].sort_values(
    ["track", "row_type", "year"], na_position="first"
)

integer_columns = [
    "year",
    "year_count",
    "batch_count",
    "wet_records_scanned",
    "wet_validated_hits",
    "warc_validated_hits",
    "final_documents",
    "target_documents",
    "adhd_documents",
    "autism_documents",
    "throughput_observed_wet_records_scanned",
    "throughput_observed_year_count",
]
for column in integer_columns:
    collection_reporting[column] = pd.to_numeric(collection_reporting[column], errors="coerce").astype("Int64")

float_columns = [
    "total_elapsed_hours",
    "hours_per_million_wet_records",
    "wet_elapsed_hours",
    "warc_elapsed_hours",
    "document_quality_elapsed_hours",
    "warc_fetch_success_rate_pct",
    "warc_extract_success_rate_pct",
]
for column in float_columns:
    collection_reporting[column] = pd.to_numeric(collection_reporting[column], errors="coerce").round(3)

REPORTING_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
collection_reporting.to_csv(REPORTING_CSV_PATH, index=False)

collection_reporting

,row_type,track,year,year_count,year_range,batch_count,instance_type,wet_records_scanned,wet_validated_hits,warc_validated_hits,final_documents,target_documents,adhd_documents,autism_documents,total_elapsed_hours,hours_per_million_wet_records,throughput_observed_wet_records_scanned,throughput_observed_year_count,throughput_complete,wet_elapsed_hours,warc_elapsed_hours,document_quality_elapsed_hours,warc_fetch_success_rate_pct,warc_extract_success_rate_pct,report_note
1,track_total,corpus,<NA>,13,2014-2026,54,m7i-flex.large,109973847,676405,315410,167520,43379,15620,33675,89.105,0.934,95374312,11,False,18.775,48.909,21.421,99.783,42.654,Quality-gated corpus track.
15,year,corpus,2014,<NA>,<NA>,3,m7i-flex.large,8222757,50481,22841,12124,3360,1121,2633,8.052,0.979,8222757,1,True,1.088,4.360,2.603,99.942,43.566,Annual accepted collection output.
16,year,corpus,2015,<NA>,<NA>,4,m7i-flex.large,10531523,56985,25663,13534,3613,1124,2892,8.557,0.813,10531523,1,True,1.399,4.755,2.403,99.907,43.108,Annual accepted collection output.
17,year,corpus,2016,<NA>,<NA>,3,m7i-flex.large,8768306,68499,32591,17471,4015,1242,3197,10.856,1.238,8768306,1,True,1.780,4.607,4.468,99.893,46.352,Annual accepted collection output.
18,year,corpus,2017,<NA>,<NA>,4,m7i-flex.large,8871512,53277,24402,12693,3262,1122,2599,7.883,0.889,8871512,1,True,1.976,4.025,1.881,99.927,44.893,Annual accepted collection output.
19,year,corpus,2018,<NA>,<NA>,5,m7i-flex.large,11816293,70241,31262,15738,4075,1347,3214,9.906,0.838,11816293,1,True,2.269,5.499,2.137,99.802,42.895,Annual accepted collection output.
20,year,corpus,2019,<NA>,<NA>,3,m7i-flex.large,6562726,42722,20805,10618,3273,1292,2611,NaN,NaN,<NA>,0,False,NaN,NaN,NaN,NaN,NaN,Annual accepted collection output.
21,year,corpus,2020,<NA>,<NA>,3,m7i-flex.large,7450904,49960,21521,10895,2917,1114,2274,8.229,1.104,7450904,1,True,1.723,4.898,1.607,99.918,42.654,Annual accepted collection output.
22,year,corpus,2021,<NA>,<NA>,4,m7i-flex.large,9660307,62686,27911,14629,3702,1323,2881,9.395,0.973,9660307,1,True,2.152,5.341,1.903,99.905,43.834,Annual accepted collection output.
23,year,corpus,2022,<NA>,<NA>,4,m7i-flex.large,8450550,53599,24292,13497,3368,1298,2527,7.959,0.942,8450550,1,True,1.833,4.569,1.556,99.783,44.883,Annual accepted collection output.
